In [4]:
import numpy as np
from pathlib import Path
import warnings

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# TensorFlow for NN surrogate (F7/F8)
import tensorflow as tf
from tensorflow import keras
tf.get_logger().setLevel('ERROR')  # Suppress TF warnings

DATA_DIR = Path("initial_data")

def suggest_ucb_point(
    X,
    y,
    beta=1.5,
    n_candidates=10_000,
    random_state=0,
    bias_point=None,
    bias_scale=0.1,
    bias_fraction=0.5
):
    """
    Given current data (X, y) for ONE function, suggest the next query point x* using
    a Gaussian Process + Upper Confidence Bound (UCB) heuristic.

    Parameters
    ----------
    X : np.ndarray, shape (N, d)
        The inputs already tried for this function.
        N = number of past points, d = input dimension (2, 3, 4, ..., 8 here).
    y : np.ndarray, shape (N,)
        The outputs observed for those inputs (same order as rows in X).
    beta : float
        Controls exploration vs exploitation in UCB = mean + beta * std.
        - Larger beta = more exploration (try uncertain points).
        - Smaller beta = more exploitation (stay near current best).
    n_candidates : int
        Base number of random candidate points to try in [0, 1]^d.
        We'll possibly increase this for higher dimensions.
    random_state : int
        Seed for the random number generator so results are reproducible.
    bias_point : np.ndarray or None
        If provided, sample some candidates near this point (for exploitation).
    bias_scale : float
        Standard deviation for biased sampling around bias_point.
    bias_fraction : float
        Fraction of candidates to sample near bias_point (rest are uniform).

    Returns
    -------
    best_x : np.ndarray, shape (d,)
        The chosen next input point (in [0, 1]^d) to submit as query.
    gp : GaussianProcessRegressor
        The fitted GP model, in case you want to inspect predictions later.
    """

    # --------------------------
    # 1. Basic info about X, y
    # --------------------------
    # X has shape (N, d): N = number of rows (past points), d = dimensions
    N, d = X.shape

    # --------------------------
    # 2. Build the GP kernel
    # --------------------------
    # RBF kernel:
    #   - Think of this as encoding the idea that nearby points in input space
    #     should have similar outputs (smoothness).
    #   - length_scale=np.ones(d) means we start by assuming each dimension
    #     has a length scale of 1.0; the GP can adapt this when fitting.
    #
    # WhiteKernel:
    #   - Adds some noise to the diagonal of the covariance matrix.
    #   - This allows the GP to handle noisy observations instead of trying
    #     to pass exactly through every point.
    kernel = 1.0 * RBF(length_scale=np.ones(d)) + WhiteKernel(noise_level=1e-3)

    # --------------------------
    # 3. Create and fit the GP
    # --------------------------
    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,   # Centers/scales y internally, helpful when magnitudes differ
        random_state=random_state,
    )

    # Fit the GP on the current data (this is where it "learns" the function shape)
    gp.fit(X, y)

    # --------------------------
    # 4. Generate candidate points
    # --------------------------
    # We'll sample random points uniformly within [0, 1]^d.
    # For higher dimensions we increase the number of candidates because
    # the space is bigger and more "empty".
    if d <= 4:
        n = n_candidates
    else:
        # Double for d >= 5 to explore a bit more widely
        n = n_candidates * 2

    # modern NumPy RNG – local, reproducible
    rng = np.random.default_rng(random_state)

    if bias_point is not None:
        # Split candidates: some near bias_point, rest uniform
        n_biased = int(n * bias_fraction)
        n_uniform = n - n_biased
        
        # Biased candidates (normal distribution around bias_point)
        Xcand_biased = rng.normal(loc=bias_point, scale=bias_scale, size=(n_biased, d))
        Xcand_biased = np.clip(Xcand_biased, 0, 1)  # Keep in [0, 1]^d
        
        # Uniform candidates
        Xcand_uniform = rng.uniform(0.0, 1.0, size=(n_uniform, d))
        
        # Combine
        Xcand = np.vstack([Xcand_biased, Xcand_uniform])
    else:
        # Xcand has shape (n, d), with each coordinate between 0 and 1
        Xcand = rng.uniform(0.0, 1.0, size=(n, d))


    # --------------------------
    # 5. GP predictions at candidates
    # --------------------------
    # For each candidate point, we ask the GP for:
    #   - mu: predicted mean (what y value we expect)
    #   - std: predictive standard deviation (how uncertain we are)
    mu, std = gp.predict(Xcand, return_std=True)

    # --------------------------
    # 6. Compute UCB score
    # --------------------------
    # UCB(x) = mu(x) + beta * std(x)
    #   - mu(x): exploitation (high mean is good)
    #   - std(x): exploration (high uncertainty is good)
    # beta trades between them.
    ucb = mu + beta * std

    # I found that we had cases where the GP would suggest points that were
    # very close to points that were already evaluated. This was a problem
    # because the GP would then suggest the same point multiple times in a row.
    # This is a simple fix: avoid re-using any already-evaluated points.
    # Vectorized version: compute, in one NumPy broadcasted operation, which
    # candidate points are (within tolerance) equal to ANY seen point, instead
    # of looping over each seen point in Python. This removes Python-loop
    # overhead, scans the candidates once, and makes the intent ("mask any
    # candidate that matches any seen point") clearer.
    same_point = np.all(
        np.isclose(Xcand[:, None, :], X[None, :, :], atol=1e-8),
        axis=2,
    )  # shape: (n_candidates, N)
    mask_any_seen = np.any(same_point, axis=1)  # shape: (n_candidates,)
    ucb[mask_any_seen] = -np.inf

    # --------------------------
    # 7. Choose the best candidate
    # --------------------------
    # Take the candidate with the highest UCB value.
    best_idx = np.argmax(ucb)
    best_x = Xcand[best_idx]  # shape (d,)

    return best_x, gp


In [5]:
# ======================================
# Neural Network Surrogate (TensorFlow)
# ======================================
# For high-dimensional functions (F7, F8), we use a small NN to propose
# candidates via gradient ascent, then let the GP decide via UCB.
# This combines NN expressiveness with GP uncertainty quantification.

def build_nn_surrogate(input_dim):
    """
    Build a small, regularized MLP surrogate model.
    Small network + dropout to avoid overfitting on tiny data.
    """
    model = keras.Sequential([
        keras.layers.Input(shape=(input_dim,)),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dropout(0.1),
        keras.layers.Dense(16, activation='relu'),
        keras.layers.Dropout(0.1),
        keras.layers.Dense(1)
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01),
                  loss='mse')
    return model


def train_nn_surrogate(X, y, epochs=500, verbose=0):
    """
    Train the NN surrogate on observed data.
    Returns the trained model and normalization parameters.
    """
    d = X.shape[1]
    
    # Normalize y for better training
    y_mean, y_std = y.mean(), y.std() + 1e-8
    y_normalized = (y - y_mean) / y_std
    
    model = build_nn_surrogate(d)
    model.fit(X, y_normalized, epochs=epochs, verbose=verbose)
    
    return model, y_mean, y_std


def gradient_ascent_on_nn(model, start_point, n_steps=100, lr=0.01):
    """
    Run gradient ascent on the NN surrogate to find a better point.
    Uses TensorFlow's GradientTape to compute gradients of output w.r.t. input.
    """
    # Start from the given point
    x = tf.Variable(start_point.reshape(1, -1), dtype=tf.float32)
    
    for _ in range(n_steps):
        with tf.GradientTape() as tape:
            y_pred = model(x, training=False)
        
        # Gradient of predicted output w.r.t. input
        grad = tape.gradient(y_pred, x)
        
        # Gradient ascent step (move in direction of increasing output)
        x.assign_add(lr * grad)
        
        # Clip to stay in [0, 1]^d
        x.assign(tf.clip_by_value(x, 0.0, 1.0))
    
    return x.numpy().flatten()


def suggest_nn_gp_hybrid(
    X,
    y,
    beta=1.5,
    n_random=1000,
    n_gradient_steps=100,
    gradient_lr=0.01,
    top_k=3,
    random_state=0
):
    """
    NN proposes candidates via gradient ascent, GP selects via UCB.
    
    Strategy:
    1. Train a small NN surrogate on the data
    2. Run gradient ascent from top-k best points → get NN-guided candidates
    3. Add random candidates as safety net
    4. GP scores ALL candidates using UCB
    5. Return the candidate with highest GP-UCB score
    
    This combines:
    - NN expressiveness for high-D gradient guidance
    - GP uncertainty for principled exploration/exploitation
    """
    N, d = X.shape
    rng = np.random.default_rng(random_state)
    
    # --------------------------
    # Step 1: Train NN surrogate
    # --------------------------
    tf.random.set_seed(random_state)
    model, y_mean, y_std = train_nn_surrogate(X, y, epochs=500, verbose=0)
    
    # --------------------------
    # Step 2: Gradient ascent from top-k points
    # --------------------------
    top_k_indices = np.argsort(y)[-top_k:]
    nn_candidates = []
    
    for idx in top_k_indices:
        start_point = X[idx].copy()
        optimized_point = gradient_ascent_on_nn(
            model, start_point, n_steps=n_gradient_steps, lr=gradient_lr
        )
        nn_candidates.append(optimized_point)
    
    nn_candidates = np.array(nn_candidates)
    
    # --------------------------
    # Step 3: Add random candidates (safety net)
    # --------------------------
    random_candidates = rng.uniform(0.0, 1.0, size=(n_random, d))
    
    # --------------------------
    # Step 4: Combine all candidates
    # --------------------------
    all_candidates = np.vstack([nn_candidates, random_candidates])
    
    # --------------------------
    # Step 5: GP scores all candidates via UCB
    # --------------------------
    kernel = 1.0 * RBF(length_scale=np.ones(d)) + WhiteKernel(noise_level=1e-3)
    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        random_state=random_state,
    )
    gp.fit(X, y)
    
    mu, std = gp.predict(all_candidates, return_std=True)
    ucb = mu + beta * std
    
    # Avoid re-using evaluated points
    same_point = np.all(
        np.isclose(all_candidates[:, None, :], X[None, :, :], atol=1e-8),
        axis=2,
    )
    mask_any_seen = np.any(same_point, axis=1)
    ucb[mask_any_seen] = -np.inf
    
    # --------------------------
    # Step 6: Return best according to GP
    # --------------------------
    best_idx = np.argmax(ucb)
    best_x = all_candidates[best_idx]
    
    # Track where the best candidate came from
    if best_idx < len(nn_candidates):
        source = f"NN gradient ascent (from top-{best_idx + 1} point)"
    else:
        source = "Random candidate"
    
    return best_x, gp, source


In [7]:
# ================================
# Load multi-round inputs/outputs
# ================================

def parse_multiline_lists(filepath):
    """
    Parse a file where each list may span multiple lines.
    Lists start with '[array(' or '[np.' and end with ')]'.
    """
    with open(filepath) as f:
        content = f.read()
    
    # Join all lines and split on list boundaries
    # Each complete entry starts with '[' and ends with ')]'
    rounds = []
    buffer = ""
    bracket_count = 0
    
    for char in content:
        buffer += char
        if char == '[':
            bracket_count += 1
        elif char == ']':
            bracket_count -= 1
            if bracket_count == 0 and buffer.strip():
                rounds.append(buffer.strip())
                buffer = ""
    
    return rounds

input_lines = parse_multiline_lists("inputs_m14.txt")
output_lines = parse_multiline_lists("outputs_m14.txt")

inputs_rounds = [eval(line, {"np": np, "array": np.array}) for line in input_lines]
outputs_rounds = [eval(line, {"np": np}) for line in output_lines]

n_rounds = len(inputs_rounds)
assert len(outputs_rounds) == n_rounds, "Mismatch between input/output rounds"

# Sanity: each round must have 8 entries (functions 1–8)
for r in range(n_rounds):
    assert len(inputs_rounds[r]) == 8, f"Round {r} inputs != 8"
    assert len(outputs_rounds[r]) == 8, f"Round {r} outputs != 8"

print(f"Loaded data from {n_rounds} previous week(s)")

# ============================
# Week 4 Configuration
# ============================

# Beta values per function
# - F1: Manual corner search (beta not used)
# - F2: Explore more after bad W3 result
# - F3: Gentler F5-style exploitation
# - F4: Local refinement around best
# - F5: Keep exploiting (working well)
# - F6: Light refinement
# - F7/F8: NN+GP hybrid
beta_map = {
    1: None,  # Manual corner search - not using GP
    2: 2.0,   # Explore more after bad W3
    3: 1.5,   # Gentler F5-style
    4: 1.0,   # Local refinement
    5: 0.5,   # Keep exploiting (working well)
    6: 1.0,   # Light refinement
    7: 1.5,   # NN+GP hybrid
    8: 1.5,   # NN+GP hybrid
}

# Bias fraction per function (for biased sampling around best point)
# - Higher = more candidates near best point (exploitation)
# - Lower = more uniform sampling (exploration)
bias_fraction_map = {
    1: None,  # Manual
    2: 0.3,   # Light bias
    3: 0.5,   # Moderate bias
    4: 0.4,   # Moderate bias
    5: 0.7,   # Heavy bias (unchanged from W3)
    6: 0.3,   # Light bias
    7: None,  # NN+GP hybrid
    8: None,  # NN+GP hybrid
}

print(f"\n{'='*60}")
print(f"WEEK {n_rounds + 1} QUERIES")
print(f"{'='*60}")

for i in range(1, 9):
    # 1. Load initial data
    X_init = np.load(DATA_DIR / f"function_{i}" / "initial_inputs.npy")
    y_init = np.load(DATA_DIR / f"function_{i}" / "initial_outputs.npy")

    X = X_init.copy()
    y = y_init.copy()

    # 2. Append every round's (x, y) for this function
    for r in range(n_rounds):
        x_prev = np.array(inputs_rounds[r][i - 1])
        y_prev = float(outputs_rounds[r][i - 1])

        print(f"Week {r + 1} {x_prev} -> {y_prev}")

        X = np.vstack([X, x_prev.reshape(1, -1)])
        y = np.append(y, y_prev)

    # 3. Find best point so far
    best_idx = np.argmax(y)
    best_point = X[best_idx]
    best_y_so_far = y[best_idx]

    # 4. Generate query based on function-specific strategy
    if i == 1:
        # --------------------------
        # F1: Manual corner search
        # --------------------------
        # F1 (contamination detection) appears flat because peaks are very localized.
        # GP can't find them. We systematically probe corners instead.
        best_x = np.array([0.05, 0.05])
        strategy = "Manual corner probe [0.05, 0.05]"
    
    elif i in [7, 8]:
        # --------------------------
        # F7, F8: NN + GP hybrid
        # --------------------------
        # High-dimensional functions where GP struggles with coverage.
        # NN proposes candidates via gradient ascent, GP selects via UCB.
        best_x, gp, source = suggest_nn_gp_hybrid(
            X, y,
            beta=beta_map[i],
            n_random=1000,
            n_gradient_steps=100,
            gradient_lr=0.01,
            top_k=3,
            random_state=40 + i,
        )
        strategy = f"NN+GP hybrid ({source})"
    
    else:
        # --------------------------
        # F2-F6: GP-UCB with biased sampling
        # --------------------------
        # Standard approach with function-specific beta and bias fraction.
        best_x, gp = suggest_ucb_point(
            X, y,
            beta=beta_map[i],
            n_candidates=10_000,
            random_state=40 + i,
            bias_point=best_point,
            bias_scale=0.1,
            bias_fraction=bias_fraction_map[i],
        )
        strategy = f"GP-UCB (β={beta_map[i]}, bias={bias_fraction_map[i]})"

    # 5. Format new query for the portal
    query_str = "-".join(f"{v:.6f}" for v in best_x)

    # 6. Report
    print(f"Function {i}:")
    print(f"  Total points: {len(y)}")
    print(f"  Strategy: {strategy}")
    print(f"  Best y so far: {best_y_so_far:.6f}")
    print(f"  New query to submit: {query_str}")
    print()

Loaded data from 3 previous week(s)

WEEK 4 QUERIES
Week 1 [0.954151 0.767932] -> 9.14184425020275e-88
Week 2 [0.125971 0.826988] -> -7.249963615484764e-176
Week 3 [0.849072 0.349783] -> -4.116065399436474e-89
Function 1:
  Total points: 13
  Strategy: Manual corner probe [0.05, 0.05]
  Best y so far: 0.000000
  New query to submit: 0.050000-0.050000

Week 1 [0.773956 0.438878] -> 0.33661711576593556
Week 2 [0.858598 0.697368] -> 0.5919902522467393
Week 3 [0.094177 0.975622] -> -0.05158916960088751
Function 2:
  Total points: 13
  Strategy: GP-UCB (β=2.0, bias=0.3)
  Best y so far: 0.611205
  New query to submit: 0.733108-0.822566

Week 1 [0.652299 0.043775 0.02003 ] -> -0.14950216326847035
Week 2 [0.839213 0.587143 0.224705] -> -0.12367642933844279
Week 3 [0.751792 0.263692 0.419978] -> -0.045964393793036476
Function 3:
  Total points: 18
  Strategy: GP-UCB (β=1.5, bias=0.5)
  Best y so far: -0.034835
  New query to submit: 0.517004-0.679411-0.281623

Week 1 [0.372882 0.447822 0.34730